In [1]:
"""
run_gtm_net.py ── GTM-KAN-MoA Entry Point
=============================================================
Usage:
    python run_gtm_net.py --dataset luo
    python run_gtm_net.py --dataset new
    python run_gtm_net.py --dataset both

SMILES and FASTA sequences are fetched automatically from PubChem and UniProt
the first run, then cached in gtmnet/seq_cache/ for all subsequent runs.
"""

import os
import json
import time
import argparse
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from data_loader import (
    load_luo_dataset, load_new_dataset, load_luo_folds,
    _ROOT, OUT_DIR
)
from gtm_net import GTMNet


def _parse_args():
    parser = argparse.ArgumentParser(description="GTM-KAN-MoA: Advanced DTI framework")
    parser.add_argument('--dataset', choices=['luo', 'new', 'both'], default='luo')
    parser.add_argument('--n_iter', type=int, default=30,
                        help="GTM EM iterations (default: 30)")
    args, _ = parser.parse_known_args()
    return args


def print_table(summary: dict, ds: str) -> None:
    print(f"\n{'=' * 56}\n  RESULTS — {ds.upper()}\n{'=' * 56}")
    print(f"  {'Metric':<10} {'Mean':>8} {'±Std':>8}")
    print("  " + "-" * 30)
    for metric in ['auroc', 'aupr', 'acc', 'prec', 'rec', 'f1']:
        if metric in summary and 'mean' in summary[metric]:
            m = summary[metric]['mean']
            s = summary[metric]['std']
            print(f"  {metric.upper():<10} {m:>8.4f} {s:>8.4f}")
    print('=' * 56)


def main():
    args = _parse_args()
    datasets = ['luo', 'new'] if args.dataset == 'both' else [args.dataset]

    print(f"\n[Environment]")
    print(f"  ROOT DIR : {_ROOT}")
    print(f"  OUT DIR  : {OUT_DIR}")

    for ds in datasets:
        print(f"\n{'#'*60}\n# GTM-KAN-MoA | Dataset: {ds.upper()}\n{'#'*60}")
        t0 = time.time()

        # Load raw network matrices + inject drug_ids / prot_ids
        data = load_luo_dataset() if ds == 'luo' else load_new_dataset()

        # Build model — sequences are resolved automatically inside fit()
        model = GTMNet(n_iter=args.n_iter, verbose=True)
        model.fit(data)

        if ds == 'luo':
            folds   = load_luo_folds()
            summary = model.evaluate_cv(folds, is_luo=True)
        else:
            from sklearn.model_selection import train_test_split
            summary = model.evaluate_random_split(n_splits=5) \
                if hasattr(model, 'evaluate_random_split') \
                else model.evaluate_cv(load_luo_folds(1), is_luo=False)

        elapsed = time.time() - t0
        print(f"\n  Total runtime: {elapsed:.1f}s")
        print_table(summary, ds)

        out_j = os.path.join(OUT_DIR, f'gtm_kan_moa_{ds}_results.json')
        with open(out_j, 'w') as fh:
            json.dump({k: v for k, v in summary.items()
                       if isinstance(v, dict) and 'mean' in v}, fh, indent=2)
        print(f"  Results saved → {out_j}")


if __name__ == '__main__':
    main()



[Environment]
  ROOT DIR : f:\Downloads\GTM_NET\GSRF-DTI-main
  OUT DIR  : f:\Downloads\GTM_NET\GSRF-DTI-main\gtmnet\results

############################################################
# GTM-KAN-MoA | Dataset: LUO
############################################################
Loading Luo dataset (raw matrices, no pre-processing) …
  [WARNING] File not found: f:\Downloads\GTM_NET\GSRF-DTI-main\data\sevenNets\mat_drug_se.txt  (fallback: random)
  [WARNING] File not found: f:\Downloads\GTM_NET\GSRF-DTI-main\data\sevenNets\mat_drug_disease.txt  (fallback: random)
  [WARNING] File not found: f:\Downloads\GTM_NET\GSRF-DTI-main\data\sevenNets\mat_protein_disease.txt  (fallback: random)

=== PHASE 1: GTM on Raw Networks ===
  S1  N= 708  M= 100  occ=50/100
  S2  N= 708  M= 100  occ=73/100
  S3  N= 708  M= 100  occ=74/100
  S4  N= 708  M= 100  occ=50/100
  S5  N=1512  M= 100  occ=38/100
  S6  N=1512  M= 100  occ=86/100
  S7  N=1512  M= 100  occ=38/100

=== PHASE 2: Joint Disease GTM ===
  Join

Loading weights: 100%|██████████| 566/566 [00:00<00:00, 6375.62it/s]
EsmModel LOAD REPORT from: facebook/esm2_t33_650M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
pooler.dense.bias           | MISSING    | 
pooler.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Loaded 5 real folds from f:\Downloads\GTM_NET\GSRF-DTI-main\dti/

=== PHASE 5: 5-Fold CV + KAN-MoA Training ===
checkpoint directory created: ./model
saving model version 0.0
  Fold 0 | AUROC=0.8481 AUPR=0.8362 Acc=0.7747
checkpoint directory created: ./model
saving model version 0.0
  Fold 1 | AUROC=0.8523 AUPR=0.8689 Acc=0.7839
checkpoint directory created: ./model
saving model version 0.0
  Fold 2 | AUROC=0.8994 AUPR=0.9093 Acc=0.8294
checkpoint directory created: ./model
saving model version 0.0
  Fold 3 | AUROC=0.8427 AUPR=0.8614 Acc=0.7904
checkpoint directory created: ./model
saving model version 0.0
  Fold 4 | AUROC=0.8783 AUPR=0.8878 Acc=0.8177

  Total runtime: 249761.1s

  RESULTS — LUO
  Metric         Mean     ±Std
  ------------------------------
  AUROC        0.8642   0.0214
  AUPR         0.8727   0.0247
  ACC          0.7992   0.0208
  PREC         0.8110   0.0329
  REC          0.7833   0.0355
  F1           0.7959   0.0205
  Results saved → f:\Downloads\GTM_NET\GS

In [1]:
"""
run_gtm_net.py ── GTM-KAN-MoA Entry Point
=============================================================
Usage:
    python run_gtm_net.py --dataset luo
    python run_gtm_net.py --dataset new
    python run_gtm_net.py --dataset both

SMILES and FASTA sequences are fetched automatically from PubChem and UniProt
the first run, then cached in gtmnet/seq_cache/ for all subsequent runs.
"""

import os
import json
import time
import argparse
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from data_loader import (
    load_luo_dataset, load_new_dataset, load_luo_folds,
    _ROOT, OUT_DIR
)
from gtm_net import GTMNet


def _parse_args():
    parser = argparse.ArgumentParser(description="GTM-KAN-MoA: Advanced DTI framework")
    parser.add_argument('--dataset', choices=['luo', 'new', 'both'], default='luo')
    parser.add_argument('--n_iter', type=int, default=30,
                        help="GTM EM iterations (default: 30)")
    args, _ = parser.parse_known_args()
    return args


def print_table(summary: dict, ds: str) -> None:
    print(f"\n{'=' * 56}\n  RESULTS — {ds.upper()}\n{'=' * 56}")
    print(f"  {'Metric':<10} {'Mean':>8} {'±Std':>8}")
    print("  " + "-" * 30)
    for metric in ['auroc', 'aupr', 'acc', 'prec', 'rec', 'f1']:
        if metric in summary and 'mean' in summary[metric]:
            m = summary[metric]['mean']
            s = summary[metric]['std']
            print(f"  {metric.upper():<10} {m:>8.4f} {s:>8.4f}")
    print('=' * 56)


def main():
    args = _parse_args()
    datasets = ['luo', 'new'] if args.dataset == 'both' else [args.dataset]

    print(f"\n[Environment]")
    print(f"  ROOT DIR : {_ROOT}")
    print(f"  OUT DIR  : {OUT_DIR}")

    for ds in datasets:
        print(f"\n{'#'*60}\n# GTM-KAN-MoA | Dataset: {ds.upper()}\n{'#'*60}")
        t0 = time.time()

        # Load raw network matrices + inject drug_ids / prot_ids
        data = load_luo_dataset() if ds == 'luo' else load_new_dataset()

        # Build model — sequences are resolved automatically inside fit()
        model = GTMNet(n_iter=args.n_iter, verbose=True)
        model.fit(data)

        if ds == 'luo':
            folds   = load_luo_folds()
            summary = model.evaluate_cv(folds, is_luo=True)
        else:
            from sklearn.model_selection import train_test_split
            summary = model.evaluate_random_split(n_splits=5) \
                if hasattr(model, 'evaluate_random_split') \
                else model.evaluate_cv(load_luo_folds(1), is_luo=False)

        elapsed = time.time() - t0
        print(f"\n  Total runtime: {elapsed:.1f}s")
        print_table(summary, ds)

        out_j = os.path.join(OUT_DIR, f'gtm_kan_moa_{ds}_results.json')
        with open(out_j, 'w') as fh:
            json.dump({k: v for k, v in summary.items()
                       if isinstance(v, dict) and 'mean' in v}, fh, indent=2)
        print(f"  Results saved → {out_j}")


if __name__ == '__main__':
    main()



[Environment]
  ROOT DIR : f:\Downloads\GTM_NET\GSRF-DTI-main
  OUT DIR  : f:\Downloads\GTM_NET\GSRF-DTI-main\gtmnet\results

############################################################
# GTM-KAN-MoA | Dataset: LUO
############################################################
Loading Luo dataset (raw matrices, no pre-processing) …
  [WARNING] File not found: f:\Downloads\GTM_NET\GSRF-DTI-main\data\sevenNets\mat_drug_se.txt  (fallback: random)
  [WARNING] File not found: f:\Downloads\GTM_NET\GSRF-DTI-main\data\sevenNets\mat_drug_disease.txt  (fallback: random)
  [WARNING] File not found: f:\Downloads\GTM_NET\GSRF-DTI-main\data\sevenNets\mat_protein_disease.txt  (fallback: random)

=== PHASE 1: GTM on Raw Networks ===
  S1  N= 708  M= 100  occ=50/100
  S2  N= 708  M= 100  occ=73/100
  S3  N= 708  M= 100  occ=73/100
  S4  N= 708  M= 100  occ=50/100
  S5  N=1512  M= 100  occ=38/100
  S6  N=1512  M= 100  occ=81/100
  S7  N=1512  M= 100  occ=38/100

=== PHASE 2: Joint Disease GTM ===
  Join

Loading weights: 100%|██████████| 566/566 [00:00<00:00, 11037.74it/s]
EsmModel LOAD REPORT from: facebook/esm2_t33_650M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias           | MISSING    | 
pooler.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Loaded 5 real folds from f:\Downloads\GTM_NET\GSRF-DTI-main\dti/

=== PHASE 5: 5-Fold CV + KAN-MoA Training ===
checkpoint directory created: ./model
saving model version 0.0


RuntimeError: mat1 and mat2 shapes cannot be multiplied (128x107 and 105x256)

In [ ]:
"""
run_gtm_net.py ── GTM-KAN-MoA Entry Point
=============================================================
Usage:
    python run_gtm_net.py --dataset luo
    python run_gtm_net.py --dataset new
    python run_gtm_net.py --dataset both

SMILES and FASTA sequences are fetched automatically from PubChem and UniProt
the first run, then cached in gtmnet/seq_cache/ for all subsequent runs.
"""

import os
import json
import time
import argparse
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from data_loader import (
    load_luo_dataset, load_new_dataset, load_luo_folds,
    _ROOT, OUT_DIR
)
from gtm_net import GTMNet


def _parse_args():
    parser = argparse.ArgumentParser(description="GTM-KAN-MoA: Advanced DTI framework")
    parser.add_argument('--dataset', choices=['luo', 'new', 'both'], default='luo')
    parser.add_argument('--n_iter', type=int, default=30,
                        help="GTM EM iterations (default: 30)")
    args, _ = parser.parse_known_args()
    return args


def print_table(summary: dict, ds: str) -> None:
    print(f"\n{'=' * 56}\n  RESULTS — {ds.upper()}\n{'=' * 56}")
    print(f"  {'Metric':<10} {'Mean':>8} {'±Std':>8}")
    print("  " + "-" * 30)
    for metric in ['auroc', 'aupr', 'acc', 'prec', 'rec', 'f1']:
        if metric in summary and 'mean' in summary[metric]:
            m = summary[metric]['mean']
            s = summary[metric]['std']
            print(f"  {metric.upper():<10} {m:>8.4f} {s:>8.4f}")
    print('=' * 56)


def main():
    args = _parse_args()
    datasets = ['luo', 'new'] if args.dataset == 'both' else [args.dataset]

    print(f"\n[Environment]")
    print(f"  ROOT DIR : {_ROOT}")
    print(f"  OUT DIR  : {OUT_DIR}")

    for ds in datasets:
        print(f"\n{'#'*60}\n# GTM-KAN-MoA | Dataset: {ds.upper()}\n{'#'*60}")
        t0 = time.time()

        # Load raw network matrices + inject drug_ids / prot_ids
        data = load_luo_dataset() if ds == 'luo' else load_new_dataset()

        # Build model — sequences are resolved automatically inside fit()
        model = GTMNet(n_iter=args.n_iter, verbose=True)
        model.fit(data)

        if ds == 'luo':
            folds   = load_luo_folds()
            summary = model.evaluate_cv(folds, is_luo=True)
        else:
            from sklearn.model_selection import train_test_split
            summary = model.evaluate_random_split(n_splits=5) \
                if hasattr(model, 'evaluate_random_split') \
                else model.evaluate_cv(load_luo_folds(1), is_luo=False)

        elapsed = time.time() - t0
        print(f"\n  Total runtime: {elapsed:.1f}s")
        print_table(summary, ds)

        out_j = os.path.join(OUT_DIR, f'gtm_kan_moa_{ds}_results.json')
        with open(out_j, 'w') as fh:
            json.dump({k: v for k, v in summary.items()
                       if isinstance(v, dict) and 'mean' in v}, fh, indent=2)
        print(f"  Results saved → {out_j}")


if __name__ == '__main__':
    main()



[Environment]
  ROOT DIR : c:\Users\biolab\Desktop\GTM_KAN_MOA
  OUT DIR  : c:\Users\biolab\Desktop\GTM_KAN_MOA\gtmnet\results

############################################################
# GTM-KAN-MoA | Dataset: LUO
############################################################
Loading Luo dataset (raw matrices, no pre-processing) …

=== PHASE 1: GTM on Raw Networks ===
  S1  N= 708  M= 100  occ=50/100
  S2  N= 708  M= 100  occ=32/100
  S3  N= 708  M= 100  occ=26/100
  S4  N= 708  M= 100  occ=50/100
  S5  N=1512  M= 100  occ=38/100
  S6  N=1512  M= 100  occ=29/100
  S7  N=1512  M= 100  occ=38/100

=== PHASE 2: Joint Disease GTM ===
  Joint GTM S36 M=100

=== PHASE 4b: Fetching Real SMILES (PubChem) + FASTA (UniProt) ===
  [SMILES] All 708 drugs loaded from cache.
  [FASTA] All 1512 proteins loaded from cache.
  Real SMILES : 708/708
  Real FASTA  : 1512/1512

=== PHASE 4b: Computing ECFP4 + ESM-2 MoA Embeddings ===
  [ESM-2] Loading cached embeddings from esm2_embed_95b28f6f24a5.pt
  

In [8]:
!conda activate OUR_work_GTM

# Remove any conflicting installs first
!pip uninstall -y numpy scipy scikit-learn

# Reinstall as a matched set (pick ONE method, don't mix conda+pip for these three)
!pip install --no-cache-dir numpy scipy scikit-learn

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: scipy 1.15.3
Uninstalling scipy-1.15.3:
  Successfully uninstalled scipy-1.15.3
Found existing installation: scikit-learn 1.1.3
Uninstalling scikit-learn-1.1.3:
  Successfully uninstalled scikit-learn-1.1.3
^C


In [ ]:
!conda activate OUR_work_GTM
!conda remove --force numpy scipy scikit-learn
!conda install numpy scipy scikit-learn

3 channel Terms of Service accepted



PackagesNotFoundError: The following packages are missing from the target environment:

  - scipy
  - scikit-learn




   ---------------------------------------- 0.0/12.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.9 MB ? eta -:--:--
   -----------------